# Generate a story with TalesWeaver

Loads a trained checkpoint from `weights/` and generates text from a prompt.
See `weights/README.md` for how to obtain a checkpoint.

In [1]:
import sys
from pathlib import Path

project_root = Path.cwd() if (Path.cwd() / "pyproject.toml").exists() else Path.cwd().parent
sys.path.insert(0, str(project_root))

import jax
import flax.nnx as nnx
import orbax.checkpoint as checkpoint
from jax.sharding import SingleDeviceSharding

from talesweaver import TalesWeaver, generate_story

## Load the checkpoint

In [2]:
model = TalesWeaver()

cpu_sharding = SingleDeviceSharding(jax.devices("cpu")[0])
restore_args = jax.tree_util.tree_map(
    lambda _: checkpoint.ArrayRestoreArgs(sharding=cpu_sharding),
    nnx.state(model),
)

checkpoint_path = (project_root / "weights/small_checkpoint.orbax").resolve()
checkpointer = checkpoint.PyTreeCheckpointer()
restored_state = checkpointer.restore(
    checkpoint_path, item=nnx.state(model), restore_args=restore_args
)
nnx.update(model, restored_state)

## Generate

In [3]:
print(generate_story(model, "Once upon a time", temperature=0.7, max_new_tokens=80))

Once upon a time, there was a little girl named Lily. She loved to play outside and eat. One day, Lily went to play with her friends. She was sad because she went to the park.
At the park, Lily saw a big, red ball. He wanted to wear it. She told her mom her and said, "That's a big, shiny, Lily. I will be careful."
